# Operator Unit Tests

| Test | Operator | What is checked | Expected result |
|------|----------|-----------------|------------------|
| G1   | **G**           | Gradient of linear $p$                                                                                        | Exact (machine $\varepsilon$)  |
| G2   | **G**           | Gradient of quadratic $p$                                                                                     | Error $O(\Delta x^2)$          |
| D1   | **DG**          | Composition $\approx \nabla^2$                                                                                | Error $O(\Delta x^2)$ interior |
| D2   | **D**           | Divergence of div-free field                                                                                  | Error $O(\Delta x^2)$ interior |
| D3   | **D**, **G**    | Adjoint: $\boldsymbol{u}^\top \mathbf{G} \boldsymbol{p} = \boldsymbol{p}^\top \mathbf{D} \boldsymbol{u}$    | Machine $\varepsilon$          |
| BC1  | $\mathbf{bc}_D$ | Global mass balance $\sum \mathbf{bc}_D = 0$                                                                  | Exact                          |
| BC2  | $\mathbf{bc}_L$ | Top-wall BC value $= 2U^*/\Delta y^2$                                                                         | Exact                          |
| L1   | **L**           | Laplacian of quadratic field $= 2$                                                                            | Exact interior                 |
| L2   | **L**           | Laplacian of linear field $= 0$                                                                               | Machine $\varepsilon$          |
| L3   | **L**           | Self-adjoint: $\boldsymbol{u}^\top \mathbf{L} \boldsymbol{v} = \boldsymbol{v}^\top \mathbf{L} \boldsymbol{u}$ | Machine $\varepsilon$         |
| A1   | **A**           | Uniform flow gives zero advection                                                                             | Machine $\varepsilon$          |
| A2   | **A**           | Energy: $\boldsymbol{u}^\top \mathbf{A}(\boldsymbol{u}) \approx 0$                                           | Small $(O(\Delta x^2))$        |
| CG1  | CG              | Solves Poisson problem to tolerance                                                                           | $< \text{tol}$                 |
| FULL | **T**           | $\boldsymbol{p}^\top \mathbf{T} \boldsymbol{p} > 0$ (SPD)                                                    | Positive                       |

## Imports

In [1]:
import numpy as np
import operators as ops

## Grid Setup

Shared across all tests. Uses a small grid (8×8) so tests run fast.

In [2]:
nx, ny = 8, 8        # small grid for fast unit tests
Lx, Ly = 1.0, 1.0
dx, dy = Lx / nx, Ly / ny
delta  = dx           # dx == dy for this square uniform grid

n_cells = nx * ny
n_vel   = ny * (nx - 1) + nx * (ny - 1)

# Index pointer arrays (same logic as func_gen_pointer in project_2.ipynb)
ip = np.full((nx, ny),     -1, dtype=int)
iu = np.full((nx + 1, ny), -1, dtype=int)
iv = np.full((nx, ny + 1), -1, dtype=int)

count = 0
for i in range(nx):
    for j in range(ny):
        ip[i, j] = count; count += 1

count = 0
for i in range(1, nx):       # interior vertical faces
    for j in range(ny):
        iu[i, j] = count; count += 1

count = ny * (nx - 1)  # v-DOFs follow u-DOFs in the flat velocity vector
for i in range(nx):
    for j in range(1, ny):   # interior horizontal faces
        iv[i, j] = count; count += 1

# Physical coordinates at pressure cell centers
xp = np.linspace(dx / 2, Lx - dx / 2, nx)
yp = np.linspace(dy / 2, Ly - dy / 2, ny)
XI, YJ = np.meshgrid(xp, yp, indexing='ij')   # shape (nx, ny)

## G1

In [3]:
# G1: gradient of linear pressure p = a*x + b*y
# For a linear field, finite differences are exact -> error should be machine epsilon.
# Expected: every u-face entry == a, every v-face entry == b

a, b = 3.0, 5.0

p = np.zeros(n_cells)
p[ip] = a * XI + b * YJ          # p(x,y) = a*x + b*y at each cell center

grad = ops.G(p, delta, ip, iu, iv, nx, ny, n_vel)

err_u = np.max(np.abs(grad[iu[1:nx, :ny]] - a))   # u-faces should all equal a
err_v = np.max(np.abs(grad[iv[:nx, 1:ny]] - b))   # v-faces should all equal b

print(f"G1  u-error: {err_u:.2e}  (expect ~machine ε ≈ 2e-16)")
print(f"G1  v-error: {err_v:.2e}  (expect ~machine ε ≈ 2e-16)")
assert err_u < 1e-12 and err_v < 1e-12, "G1 FAILED"
print("G1 PASSED")

G1  u-error: 0.00e+00  (expect ~machine ε ≈ 2e-16)
G1  v-error: 0.00e+00  (expect ~machine ε ≈ 2e-16)
G1 PASSED
